# Refine Diagnosis Using Medications

**Mode: `refine`** — Two-phase medication-based diagnosis refinement agent.

This notebook uses a **two-phase approach** to refine diagnoses:
- **Phase 1:** Medication-guided hierarchy drill-down via `get_medication_diagnosis_proto`
- **Phase 2:** Note-driven refinement axes via `get_allowed_refinements` + `get_narrower_sequential_refinements`

**9 tools:** `normalize_medical_term`, `get_lexical`, `get_domain_hierarchy`, `get_treatments_for_problem`, `get_caused_problems`, `get_diagnostics_for_problem`, `get_allowed_refinements`, `get_narrower_sequential_refinements`, `get_medication_diagnosis_proto`

### Requirements
```
pip install langchain-aws langgraph requests
```

Set environment variables:
```
IMO_CLIENT_ID / IMO_CLIENT_SECRET      — Normalize API
IMO_KG_CLIENT_ID / IMO_KG_CLIENT_SECRET — Knowledge Graph API
AWS credentials configured (for Bedrock)
```

In [ ]:
import sys, os, re, json, html as html_mod, uuid
from collections import defaultdict
from typing import List

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from IPython.display import display, HTML, update_display

os.environ["IMO_NORMALIZE_CLIENT_ID"] = ""
os.environ["IMO_NORMALIZE_SECRET"]    = ""
os.environ["IMO_KG_CLIENT_ID"]        = ""
os.environ["IMO_KG_CLIENT_SECRET"]    = ""

os.environ["AWS_ACCESS_KEY_ID"]     = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"]     = ""
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import kg_config as kc
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from kg_api_client import KGApiClient

print("✓  Credentials loaded")

✓  Credentials loaded


## LLM, Tools & Agent Setup

All 9 tools from the `refine_with_kg` route's `refine` mode, with token overflow prevention and conversation history.

In [2]:
_kg_client = KGApiClient()

_KG_LIST_THRESHOLD = 200
_MAX_TOOL_RESPONSE_CHARS = 100000

def _truncate_kg_lists(obj, path=""):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if isinstance(v, list) and len(v) > _KG_LIST_THRESHOLD:
                out[k] = v[:_KG_LIST_THRESHOLD]
                out[f"_{k}_summary"] = f"Showing {_KG_LIST_THRESHOLD} of {len(v)} total items (truncated to prevent token overflow)"
            else:
                out[k] = _truncate_kg_lists(v, f"{path}.{k}")
        return out
    elif isinstance(obj, list):
        return [_truncate_kg_lists(item, f"{path}[]") for item in obj]
    return obj

def _aggressive_truncate(text, max_chars):
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f'\n... [TRUNCATED at {max_chars} chars of {len(text)} total]'

def _dumps(obj):
    return json.dumps(obj, indent=2)

def _cap_tool_response(result):
    truncated = _truncate_kg_lists(result)
    text = _dumps(truncated)
    return _aggressive_truncate(text, _MAX_TOOL_RESPONSE_CHARS)


# ─── 9 Tools ────────────────────────────────────────────────────────────────

@tool
def normalize_medical_term(input_term: str, domain: str = "Medication", order_name: str = "") -> str:
    """Normalize a medical term using IMO Precision Normalize API.

    Args:
        input_term: The clinical term (e.g., "Metformin", "diabetes mellitus", "glucose"). Must not be empty.
        domain: "Medication" for drugs, "Problem" for diagnoses, "Lab" for lab results. Default is Medication.
        order_name: Optional lab order name (e.g., "lipid panel"). Only used when domain="Lab".
    """
    if not input_term or not input_term.strip():
        return json.dumps({"success": False, "error": "STOP: input_term is empty. Do NOT retry with empty string. Do NOT call this tool again without a valid medication name. Skip and proceed to scoring."})
    result = _kg_client.normalize_medical_term(input_term.strip(), domain, order_name=order_name.strip() if order_name else "")
    return _dumps(result)


@tool
def get_lexical(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Query the IMO Knowledge Graph for a concept's core data.
    - For medication (domain="medication"): treatedProblems, causedProblems
    - For problem (domain="problem"): allowedRefinements, narrower subtypes, refinementFamilies, mappings

    NOTE: For problem treatments/contraindications, use get_treatments_for_problem instead.
    NOTE: Large result lists (500+ items) are automatically truncated.

    Args:
        imo_lexical_code: The default_lexical_code from normalize results. REQUIRED.
        domain: "problem" or "medication"
    """
    if not imo_lexical_code or not imo_lexical_code.strip():
        return json.dumps({"success": False, "error": "STOP: imo_lexical_code is empty. You must provide a valid lexical code from normalize results."})
    result = _kg_client.get_lexical(imo_lexical_code.strip(), domain)
    return _cap_tool_response(result)


@tool
def get_domain_hierarchy(imo_lexical_code: str, direction: str = "narrower", domain: str = "Problem") -> str:
    """Get domain hierarchy (children or parents) for a problem concept.
    - direction="narrower": Get subtypes/children
    - direction="broader": Get parent concepts

    Also returns synonyms for the concept.
    NOTE: Large result lists (500+ items) are automatically truncated.

    Args:
        imo_lexical_code: The IMO lexical code. REQUIRED.
        direction: "narrower" for children/subtypes, "broader" for parent concepts
        domain: Use "Problem"
    """
    if not imo_lexical_code or not imo_lexical_code.strip():
        return json.dumps({"success": False, "error": "STOP: imo_lexical_code is empty. You must provide a valid lexical code."})
    result = _kg_client.get_domain_hierarchy(imo_lexical_code.strip(), direction, domain)
    return _cap_tool_response(result)


@tool
def get_treatments_for_problem(imo_lexical_code: str) -> str:
    """Get treatment-related medications for a problem/diagnosis from the KG.
    Returns: associatedTreatments, contraindicatedTreatments, supportiveTreatments.

    Args:
        imo_lexical_code: The default_lexical_code of the problem (from normalize with domain="Problem")
    """
    result = _kg_client.get_treatments_for_problem(imo_lexical_code)
    return _cap_tool_response(result)


@tool
def get_caused_problems(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Get problems/complications caused by a condition or medication.
    - For problem: returns causedProblems (complications) and causativeAgents
    - For medication: returns causedProblems (adverse effects)

    Args:
        imo_lexical_code: The default_lexical_code from normalize results
        domain: "problem" or "medication"
    """
    result = _kg_client.get_caused_problems(imo_lexical_code, domain)
    return _cap_tool_response(result)


@tool
def get_diagnostics_for_problem(imo_lexical_code: str) -> str:
    """Get diagnostic tests and lab procedures relevant to a problem/diagnosis.
    Returns: associatedProcedures.

    Args:
        imo_lexical_code: The default_lexical_code of the problem (from normalize with domain="Problem")
    """
    result = _kg_client.get_diagnostics_for_problem(imo_lexical_code)
    return _cap_tool_response(result)


@tool
def get_allowed_refinements(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Get allowed refinements for a problem concept. Returns refinement groups
    (type, laterality, severity, chronicity, etc.) that can further narrow a diagnosis.

    Args:
        imo_lexical_code: The IMO lexical code of the problem
        domain: Use "Problem".
    """
    result = _kg_client.get_allowed_refinements(imo_lexical_code, domain)
    return _dumps(result)


@tool
def get_narrower_sequential_refinements(
    imo_lexical_code: str,
    refinement_sequence: List[List[str]],
    include_mappings: bool = True,
    domain: str = "Problem",
) -> str:
    """Apply refinements sequentially to resolve the most specific concept.

    Args:
        imo_lexical_code: Base concept lexical code
        refinement_sequence: Array of arrays, each inner array is one refinement step. Example: [["230"], ["362636588"]]
        include_mappings: Include ICD-10 mappings in response
        domain: Use "Problem".
    """
    result = _kg_client.get_narrower_sequential_refinements(
        imo_lexical_code, refinement_sequence, include_mappings, domain
    )
    return _dumps(result)


@tool
def get_medication_diagnosis_proto(
    diagnosis_code: str,
    medication_codes: List[str],
) -> str:
    """KG lookup: return narrower problem concepts of a base diagnosis that have ties
    to the supplied medication codes, via domainNarrowerByMedications.

    Use this for the medication-guided hierarchy drill-down (Phase 1).
    Results are sorted with non-leaf concepts (numberOfDomainChildren > 0) first.

    Args:
        diagnosis_code: Base diagnosis lexical code (from normalize with domain="Problem")
        medication_codes: List of medication lexical codes (from normalize with domain="Medication")
    """
    result = _kg_client.get_medication_diagnosis_proto(diagnosis_code, medication_codes)
    if result.get("success") and result.get("lexical"):
        items = result["lexical"].get("domainNarrowerByMedications", [])
        items.sort(key=lambda x: (x.get("numberOfDomainChildren", 0) == 0, x.get("title", "")))
        result["lexical"]["domainNarrowerByMedications"] = items
    return _cap_tool_response(result)


TOOLS = [
    normalize_medical_term,
    get_lexical,
    get_domain_hierarchy,
    get_treatments_for_problem,
    get_caused_problems,
    get_diagnostics_for_problem,
    get_allowed_refinements,
    get_narrower_sequential_refinements,
    get_medication_diagnosis_proto,
]

print(f"KG client initialized. {len(TOOLS)} tools registered.")

KG client initialized. 9 tools registered.


## System Prompt

Two-phase medication-based diagnosis refinement with EXPLICIT DOCUMENTATION RULE (categories A/B/C/D).

In [3]:
SYSTEM_PROMPT = """You are a medication-based diagnosis refinement agent. You refine diagnoses to their most specific form using a two-phase approach: medication-guided hierarchy drill-down followed by note-driven refinement axes.

## WORKFLOW

### Step 1: Extract (NO TOOL CALLS)
From the clinical note, extract:
- Base diagnoses (strip ALL qualifiers — type, severity, laterality, chronicity, complications)
  Example: "chronic systolic congestive heart failure" → base = "heart failure"
  Example: "type 2 diabetes mellitus with nephropathy" → base = "diabetes mellitus"
- Medications associated with each diagnosis
- Clinical qualifiers from the note in PRIORITY ORDER:
  1. Type/category (e.g., type 2, primary, systolic)
  2. Chronicity (e.g., chronic, acute, acute on chronic)
  3. Complications/manifestations (e.g., nephropathy, retinopathy, congestive)
  4. Severity/stage (e.g., moderate, stage 3, NYHA class 3)
  5. Laterality (e.g., left, right, bilateral)
  6. Other qualifiers (e.g., uncontrolled, with insulin use)

Present a brief summary of extraction, then immediately proceed to Step 2.

### Step 2: Normalize Base Diagnosis and Medications
1. Call normalize_medical_term(input_term=<base diagnosis>, domain="Problem") to get its default_lexical_code
2. For each medication, call normalize_medical_term(input_term=<med>, domain="Medication") to get its default_lexical_code
Collect all medication codes into a list.

### Step 3: PHASE 1 — Medication-Guided Hierarchy Drill-Down
Call get_medication_diagnosis_proto(diagnosis_code=<base dx code>, medication_codes=[<all med codes>]).
This returns narrower concepts of the base diagnosis that have ties to those medications.

**Iterative drill-down rules:**
1. Each returned candidate has `code`, `title`, and `numberOfDomainChildren`
2. Pick the candidate matching the HIGHEST-PRIORITY unused qualifier from Step 1 (type > chronicity > complication > severity > laterality) — BUT ONLY if that qualifier is EXPLICITLY DOCUMENTED in the note text (see EXPLICIT DOCUMENTATION RULE below)
3. Among qualifying candidates, prefer those with numberOfDomainChildren > 0 (non-leaves) over leaves
4. Apply ONE qualifier per depth level — do not jump to a concept matching 3 qualifiers at once when a single-qualifier match with children exists
5. After picking, call get_medication_diagnosis_proto AGAIN with the picked candidate's code and the SAME medication_codes list
6. STOP when:
   - numberOfDomainChildren = 0 (leaf reached), OR
   - The tool returns an empty list (no further med-linked children), OR
   - NO candidate at this level matches an EXPLICITLY DOCUMENTED qualifier (STOP HERE — proceed to Phase 2), OR
   - 8 rounds completed (safety bound)

**If Phase 1 returns EMPTY on the very first call** (no med-linked subtypes at all):
- Skip to Step 4 (Phase 2) directly — the medications don't provide hierarchy guidance for this condition, so use refinement axes instead.

### Step 4: PHASE 2 — Note-Driven Refinement Axes (ONE REFINEMENT AT A TIME)
Starting from the concept reached at the end of Phase 1 (or the base diagnosis if Phase 1 was empty):

Apply refinements ONE AT A TIME iteratively:

1. Call get_allowed_refinements on the CURRENT concept's code
2. Check which UNUSED qualifiers from Step 1 match available refinement groups. Also check medication-observable qualifiers (rule B) like "without long term use of insulin"
3. Pick the HIGHEST-PRIORITY matching refinement code
4. Call get_narrower_sequential_refinements with ONLY that single refinement: refinement_sequence=[["<code>"]]
5. If it returns a resolved concept:
   - That concept becomes your NEW current concept
   - Go back to step 1 and repeat (check allowed refinements on the new concept for remaining qualifiers)
6. If it returns EMPTY:
   - Skip that refinement and try the next matching one
   - If all remaining refinements return empty, STOP — use the current concept as final
7. STOP when no more matching refinements remain or 5 iterations completed

### Step 5: Present Results
Use normalize_medical_term on the final refined diagnosis (domain="Problem") to get ICD-10 codes, then present:

#### Medication-Based Diagnosis Refinement Results

| Original Diagnosis | Medication | Refined Diagnosis | ICD-10 | Reasoning |
|---|---|---|---|---|
| (base) | (drug) | (refined) | (code) | (why) |

For each refined diagnosis include:
- **IMO Lexical Code:** (code)
- **IMO KG Link:** https://studio.imohealth.com/#/graph-browser?id=(code)
- **Refinement Path:** Show full drill-down path:
  - Phase 1 steps: base → picked concept (reason) → picked concept (reason) → ... → leaf
  - Phase 2 steps: leaf → +refinement applied → final concept
- **Phase 1 Depth:** Number of drill-down levels used
- **Phase 2 Refinements Applied:** List of refinement axes used, or "None needed"

## RULES
- Always use default_lexical_code from normalize results for KG lookups
- NEVER call normalize with an empty string — skip any term that is empty or unclear
- Do NOT wait for user input after extraction — refine all diagnoses automatically
- NEVER fabricate ICD-10 codes — get them from normalize tool
- NEVER invent or assume a more specific diagnosis if the KG did not return data for it. If a tool returns empty/no results, use the last successfully resolved concept and clearly state "No further KG data available for this refinement"
- Do NOT use your own medical knowledge to pick a more specific code — only use what the KG tools return
- Before each tool call, briefly explain why you're calling it
- When Phase 1 returns many candidates (50+), focus on candidates matching your highest-priority unused qualifier — do not attempt to list or evaluate all of them
- CRITICAL — EXPLICIT DOCUMENTATION RULE: During Phase 1 drill-down AND Phase 2 refinement selection, you may ONLY pick a candidate/refinement whose qualifier is justified by one of the following:
  (A) EXPLICITLY WRITTEN in the note by the provider — the exact word or direct synonym appears in Assessment, HPI, or Problem List text, OR
  (B) DIRECTLY OBSERVABLE from the medication list — factual medication-profile qualifiers:
      - "without long-term current use of insulin" → valid if no insulin in med list
      - "treated with insulin" → valid if insulin IS in the med list
      - "treated with oral medication" → valid if oral diabetes meds are prescribed
  (C) TYPE/CATEGORY identification from medication class — the medication's pharmacological class directly identifies the disease subtype (e.g., Metformin is exclusively a type 2 DM drug → "type 2 diabetes mellitus" is valid). This applies ONLY to type/category distinctions, NOT to severity or control status.
  (D) VALIDATED CLINICAL ASSESSMENT TOOLS — when a disease-specific standardized assessment is documented in the note, it identifies the disease subtype:
      - PHQ-9 score documented → identifies "major depressive disorder"
      - Ejection fraction (EF%) → identifies HFrEF vs HFpEF
      - NYHA class documented → identifies functional classification
      - FEV1/FVC ratio → identifies obstructive vs restrictive lung disease
      This applies to TYPE/CATEGORY identification only, NOT to severity or control status.
  - What is NOT allowed (requires clinical inference/judgment):
    - Control status: "uncontrolled", "well controlled", "poorly controlled" — NEVER infer from labs or clinical context, must be explicitly documented by the provider
    - HbA1c values do NOT justify any control-status qualifier
    - "Continue current regimen" does NOT justify "well controlled" or "controlled"
    - PHQ-9 score does NOT justify severity level (mild/moderate/severe) unless the provider explicitly documents the severity word
  - If NO candidate at a given level has a qualifier satisfying (A), (B), (C), or (D), STOP PHASE 1 at the current level (even if non-leaf candidates exist) and proceed to Phase 2 with the current concept
  - This rule takes ABSOLUTE PRIORITY over "prefer non-leaves" — a non-leaf candidate with an inferred control-status qualifier must be REJECTED in favor of stopping

## SUMMARY BLOCK (REQUIRED)
After your final answer, include a summary block wrapped in `[SUMMARY]` and `[/SUMMARY]` tags. This is displayed in a visual explainer panel for non-technical audiences. Write in simple, plain language.

**FORMAT: Exactly 4 paragraphs separated by blank lines.** Each paragraph maps to a visual card:

1. **Paragraph 1 — Clinical Question:** Restate what the user asked in 1-2 plain sentences. Mention the patient scenario and the diagnosis being refined.
2. **Paragraph 2 — What the Agent Did:** Describe the steps: normalized the diagnosis, queried the Knowledge Graph for refinement axes, drilled down the medication-guided hierarchy, and resolved the most specific concept.
3. **Paragraph 3 — Knowledge Graph Value:** The KG provided refinement axes and a medication-guided hierarchy to drill down to the most specific diagnosis. Emphasize that these are **curated, clinically-verified relationships** that cannot be derived from text matching — this is the KG's unique value.
4. **Paragraph 4 — Recommendation:** State the final refined diagnosis and its ICD-10 code, and why it was chosen.

Keep each paragraph concise (2-4 sentences max). Bold key terms."""


llm = ChatBedrockConverse(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name="us-east-1",
    temperature=0,
    max_tokens=64000,
    provider="anthropic",
)

agent = create_react_agent(llm, TOOLS, prompt=SYSTEM_PROMPT)

print("Agent created with refine mode system prompt (two-phase refinement).")

Agent created with refine mode system prompt (two-phase refinement).


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_11020\304517309.py:131: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, TOOLS, prompt=SYSTEM_PROMPT)


## HTML Renderer + Streaming Engine

CSS ported from `chat_med_refinement.html` + Python functions that render
agent activity into rich HTML panels. Supports **live streaming** via `update_display()`.

In [4]:
def esc(text):
    return html_mod.escape(str(text)) if text else ""

AGENT_CSS = """
<style>
:root {
    --primary: #57039E; --primary-light: #7C0EDD; --primary-dark: #3D0270;
    --accent: #10b981; --danger: #ef4444;
    --bg: #f3f4f6; --bg-card: #ffffff; --text: #1e293b; --text-secondary: #64748b; --border: #e5e7eb;
}
.agent-panel { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; color:var(--text); line-height:1.6; margin:1rem 0; }
.agent-panel * { box-sizing:border-box; }
.msg-container { display:flex; gap:0.75rem; margin-bottom:1.25rem; animation:fadeIn 0.3s ease-in; }
@keyframes fadeIn { from{opacity:0;transform:translateY(8px)} to{opacity:1;transform:translateY(0)} }
.msg-avatar { width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1.2rem;flex-shrink:0;background:var(--accent); }
.msg-content { flex:1;background:var(--bg-card);border-radius:12px;padding:1rem;box-shadow:0 1px 2px rgba(0,0,0,0.05); }
.msg-header { display:flex;justify-content:space-between;align-items:center;margin-bottom:0.5rem; }
.msg-sender { font-weight:600;color:var(--text);font-size:0.9rem; }
.msg-time { font-size:0.75rem;color:var(--text-secondary); }
.msg-user .msg-avatar { background:var(--primary); }
.msg-user .msg-content { background:var(--primary);color:#fff; }
.msg-user .msg-sender,.msg-user .msg-time { color:rgba(255,255,255,0.8); }
.clinical-note { background:rgba(87,3,158,0.04);border:2px solid var(--primary);border-radius:12px;padding:1.25rem;margin-bottom:1rem; }
.clinical-note-hdr { display:flex;align-items:center;gap:0.6rem;margin-bottom:0.75rem;padding-bottom:0.5rem;border-bottom:2px solid var(--primary); }
.clinical-note-title { font-weight:700;font-size:0.95rem;color:var(--primary);text-transform:uppercase;letter-spacing:0.05em; }
.clinical-note-body { font-size:0.92rem;line-height:1.7;white-space:pre-wrap; }
.hl { background:linear-gradient(to bottom,transparent 85%,#fbbf24 85%);font-weight:600;border-bottom:2px solid #f59e0b;padding-bottom:1px; }
.reasoning-steps { margin-bottom:1rem;border:1px solid var(--border);border-radius:10px;overflow:hidden;background:var(--bg); }
.reasoning-hdr { display:flex;justify-content:space-between;align-items:center;padding:0.6rem 1rem;background:#f0e7ff;cursor:pointer;user-select:none;border-bottom:1px solid var(--border); }
.reasoning-hdr h4 { font-size:0.88rem;color:var(--primary-dark);margin:0; }
.step-count { font-size:0.68rem;font-weight:600;padding:0.1rem 0.45rem;border-radius:999px;background:var(--primary);color:#fff; }
.toggle-icon { font-size:0.7rem;color:var(--text-secondary);transition:transform 0.2s; }
.reasoning-body { padding:0.6rem;display:flex;flex-direction:column;gap:0.6rem; }
.step { padding:0.7rem 0.85rem;background:var(--bg-card);border-radius:8px;position:relative; }
.step-reasoning { border-left:4px solid #f59e0b;background:linear-gradient(135deg,#fffbeb,var(--bg-card)); }
.step-tool { border-left:4px solid #6d28d9;background:linear-gradient(135deg,#ede9fe,#f5f3ff);box-shadow:0 1px 3px rgba(109,40,217,0.1); }
.step-tool.normalize { border-left-color:#0891b2;background:linear-gradient(135deg,#ecfeff,var(--bg-card));box-shadow:none; }
.step-result { border-left:4px solid var(--accent); }
.step-badge { position:absolute;top:0.5rem;right:0.6rem;color:#fff;font-size:0.63rem;font-weight:700;padding:0.1rem 0.4rem;border-radius:999px;text-transform:uppercase; }
.badge-kg { background:#7c3aed; }
.badge-normalize { background:#0891b2; }
.badge-done { background:var(--accent); }
.step-hdr { display:flex;align-items:center;gap:0.4rem;margin-bottom:0.25rem;font-weight:600;font-size:0.88rem; }
.step-icon { width:22px;height:22px;border-radius:50%;display:inline-flex;align-items:center;justify-content:center;font-size:0.75rem;flex-shrink:0; }
.icon-tool { background:#ede9fe;color:#7c3aed; }
.icon-normalize { background:#cffafe;color:#0e7490; }
.icon-result { background:#d1fae5;color:#065f46; }
.icon-reasoning { background:#fef3c7;color:#92400e; }
.step-detail { font-size:0.83rem;color:var(--text-secondary);padding-left:1.8rem; }
.step-reasoning .step-title { color:#92400e; }
.step-reasoning .reasoning-text { font-size:0.83rem;color:var(--text);line-height:1.5;padding-left:1.8rem; }
.step-tool.normalize .step-title { color:#0e7490; }
.result-table-wrap { overflow:auto;margin-top:0.3rem;max-height:350px;border:1px solid var(--border);border-radius:6px; }
.result-table { width:100%;border-collapse:collapse;font-size:0.8rem; }
.result-table thead th { background:var(--primary);color:#fff;font-weight:600;text-align:left;padding:0.4rem 0.6rem;white-space:nowrap; }
.result-table tbody tr { border-bottom:1px solid var(--border); }
.result-table tbody tr:hover { background:#f1f5f9; }
.result-table tbody td { padding:0.35rem 0.6rem;color:var(--text); }
.raw-toggle { margin-top:0.5rem;padding-top:0.5rem;border-top:1px solid var(--border); }
.raw-toggle summary { font-size:0.75rem;font-weight:600;color:#7c3aed;cursor:pointer;padding:0.3rem 0.5rem;border-radius:6px; }
.raw-toggle summary:hover { background:rgba(87,3,158,0.05); }
.raw-toggle pre { margin:0.4rem 0 0;background:#f8f9fa;border:1px solid #e2e8f0;border-radius:6px;padding:0.6rem;max-height:300px;overflow-y:auto;font-size:0.72rem;white-space:pre-wrap;word-wrap:break-word; }
.final-answer { margin-top:0.5rem; }
.final-label { font-size:0.73rem;font-weight:700;text-transform:uppercase;letter-spacing:0.05em;color:var(--accent);margin-bottom:0.3rem;padding-bottom:0.3rem;border-bottom:2px solid var(--accent); }
.final-body { font-size:0.95rem;line-height:1.7; }
.final-body strong { color:var(--primary); }
.final-body table { width:100%;border-collapse:collapse;margin:0.5rem 0;font-size:0.85rem; }
.final-body th { background:var(--primary);color:#fff;padding:0.4rem 0.6rem;text-align:left; }
.final-body td { padding:0.35rem 0.6rem;border-bottom:1px solid var(--border); }
.final-body code { background:#f1f5f9;padding:0.1rem 0.35rem;border-radius:4px;font-size:0.85em; }
.streaming-indicator { display:flex;align-items:center;gap:0.35rem;padding:0.75rem 1rem;color:var(--primary);font-weight:600;font-size:0.88rem;background:linear-gradient(135deg,rgba(87,3,158,0.04),rgba(124,14,221,0.06));border-radius:8px;margin-top:0.5rem; }
.dot-pulse { display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--primary);animation:pulse 1.4s ease-in-out infinite; }
.dot-pulse:nth-child(2) { animation-delay:0.2s; }
.dot-pulse:nth-child(3) { animation-delay:0.4s; }
@keyframes pulse { 0%,100%{opacity:0.2;transform:scale(0.7)} 50%{opacity:1;transform:scale(1.2)} }
</style>
"""

TOOL_NAMES = {
    "normalize_medical_term": "Normalize Medical Term",
    "get_lexical": "GraphQL: Get Lexical",
    "get_domain_hierarchy": "GraphQL: Domain Hierarchy",
    "get_treatments_for_problem": "GraphQL: Treatments for Problem",
    "get_caused_problems": "GraphQL: Caused Problems",
    "get_diagnostics_for_problem": "GraphQL: Diagnostics for Problem",
    "get_allowed_refinements": "GraphQL: Allowed Refinements",
    "get_narrower_sequential_refinements": "GraphQL: Sequential Refinements",
    "get_medication_diagnosis_proto": "GraphQL: Med-Diagnosis Drill-Down",
}

def is_normalize(tool_name):
    return "normalize" in tool_name.lower()

def fmt_tool_input(tool_name, args):
    if tool_name == "normalize_medical_term":
        return f'Normalizing <strong>"{esc(args.get("input_term", ""))}"</strong> in domain <strong>{esc(args.get("domain", "Medication"))}</strong>'
    elif tool_name == "get_lexical":
        return f'Looking up lexical code <strong>{esc(args.get("imo_lexical_code", ""))}</strong> (domain: {esc(args.get("domain", "Problem"))})'
    elif tool_name == "get_domain_hierarchy":
        return f'Getting <strong>{esc(args.get("direction", "narrower"))}</strong> hierarchy for <strong>{esc(args.get("imo_lexical_code", ""))}</strong>'
    elif tool_name == "get_treatments_for_problem":
        return f'Getting treatments for problem <strong>{esc(args.get("imo_lexical_code", ""))}</strong>'
    elif tool_name == "get_caused_problems":
        return f'Getting caused problems for <strong>{esc(args.get("imo_lexical_code", ""))}</strong> (domain: {esc(args.get("domain", "Problem"))})'
    elif tool_name == "get_diagnostics_for_problem":
        return f'Getting diagnostic procedures for <strong>{esc(args.get("imo_lexical_code", ""))}</strong>'
    elif tool_name == "get_allowed_refinements":
        return f'Getting allowed refinements for <strong>{esc(args.get("imo_lexical_code", ""))}</strong>'
    elif tool_name == "get_narrower_sequential_refinements":
        return f'Resolving narrower concept for <strong>{esc(args.get("imo_lexical_code", ""))}</strong> with refinements <strong>{esc(json.dumps(args.get("refinement_sequence", [])))}</strong>'
    elif tool_name == "get_medication_diagnosis_proto":
        return f'Med-guided drill-down: diagnosis <strong>{esc(args.get("diagnosis_code", ""))}</strong> with <strong>{len(args.get("medication_codes", []))} medication(s)</strong>'
    return f'<pre>{esc(json.dumps(args, indent=2))}</pre>'

def fmt_tool_result_html(tool_name, result_str):
    try:
        data = json.loads(result_str) if isinstance(result_str, str) else result_str
    except (json.JSONDecodeError, TypeError):
        return f'<pre>{esc(str(result_str)[:500])}</pre>'

    html_out = ""
    if tool_name == "normalize_medical_term" and isinstance(data, dict) and data.get("results"):
        matches = data["results"][0].get("matches", []) if data["results"] else []
        term = data["results"][0].get("input_term", "") if data["results"] else ""
        html_out += f'<div style="font-size:0.83rem;color:var(--text-secondary);margin-bottom:0.3rem;">{len(matches)} result(s) for "<strong>{esc(term)}</strong>"</div>'
        html_out += '<div class="result-table-wrap"><table class="result-table"><thead><tr><th>IMO Description</th><th>Lexical Code</th><th>Score</th><th>ICD-10</th></tr></thead><tbody>'
        for m in matches[:5]:
            code = m.get("default_lexical_code", m.get("lexical_code", ""))
            icd = ", ".join(m.get("icd10_codes", []))
            html_out += f'<tr><td><strong>{esc(m.get("title", ""))}</strong></td><td>{esc(code)}</td><td>{m.get("score", 0):.3f}</td><td>{esc(icd)}</td></tr>'
        html_out += '</tbody></table></div>'
    elif tool_name == "get_medication_diagnosis_proto" and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        items = lex.get("domainNarrowerByMedications", [])
        html_out += f'<div style="font-size:0.83rem;">Base: <strong>{esc(lex.get("title", ""))}</strong> — {len(items)} med-linked narrower concepts</div>'
        if items:
            html_out += '<div class="result-table-wrap" style="max-height:250px;"><table class="result-table"><thead><tr><th>Title</th><th>Code</th><th>Children</th></tr></thead><tbody>'
            for it in items[:30]:
                html_out += f'<tr><td>{esc(it.get("title", ""))}</td><td>{esc(it.get("code", ""))}</td><td>{it.get("numberOfDomainChildren", 0)}</td></tr>'
            if len(items) > 30:
                html_out += f'<tr><td colspan="3"><em>... and {len(items)-30} more</em></td></tr>'
            html_out += '</tbody></table></div>'
    elif tool_name in ("get_lexical", "get_allowed_refinements") and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        html_out += f'<div style="font-size:0.83rem;">Lexical: <strong>{esc(lex.get("title", ""))}</strong> (code: {esc(lex.get("code", ""))})</div>'
        allowed = lex.get("allowedRefinements", []) or []
        if allowed:
            groups = defaultdict(list)
            for ref in allowed:
                g = (ref.get("group") or {}).get("title", "Ungrouped")
                groups[g].append(ref)
            html_out += f'<div style="margin-top:0.4rem;"><strong>Allowed Refinement Groups ({len(groups)}):</strong></div>'
            for gname, refs in sorted(groups.items()):
                html_out += f'<div style="margin:0.3rem 0 0.1rem;padding:0.3rem 0.5rem;background:#f8fafc;border:1px solid var(--border);border-radius:6px;"><strong>{esc(gname)}</strong> <span style="font-size:0.7rem;background:var(--primary);color:#fff;padding:0.1rem 0.3rem;border-radius:999px;">{len(refs)}</span></div>'
                html_out += '<div class="result-table-wrap" style="max-height:180px;"><table class="result-table"><thead><tr><th>Refinement</th><th>Code</th></tr></thead><tbody>'
                for r in refs:
                    html_out += f'<tr><td>{esc(r.get("title", ""))}</td><td>{esc(r.get("code", ""))}</td></tr>'
                html_out += '</tbody></table></div>'
    elif tool_name == "get_narrower_sequential_refinements" and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        narrower = lex.get("refinementNarrower", []) or []
        html_out += f'<div style="font-size:0.83rem;">Base: <strong>{esc(lex.get("title", ""))}</strong></div>'
        if narrower:
            html_out += f'<div><strong>Resolved ({len(narrower)}):</strong></div>'
            html_out += '<div class="result-table-wrap"><table class="result-table"><thead><tr><th>Title</th><th>Code</th><th>Applied Refinements</th></tr></thead><tbody>'
            for n in narrower:
                applied = ", ".join(r.get("title", "") for r in (n.get("appliedRefinements") or []))
                html_out += f'<tr><td><strong>{esc(n.get("title", ""))}</strong></td><td>{esc(n.get("code", ""))}</td><td>{esc(applied)}</td></tr>'
            html_out += '</tbody></table></div>'
        else:
            html_out += '<div><em>No narrower concepts resolved.</em></div>'
    elif tool_name in ("get_treatments_for_problem", "get_caused_problems", "get_diagnostics_for_problem") and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        html_out += f'<div style="font-size:0.83rem;">Concept: <strong>{esc(lex.get("title", ""))}</strong></div>'
        for field in ("associatedTreatments", "supportiveTreatments", "causedProblems", "causativeAgents", "associatedProcedures"):
            items = lex.get(field, [])
            if items:
                html_out += f'<div style="margin-top:0.3rem;"><strong>{field} ({len(items)}):</strong></div>'
                html_out += '<div class="result-table-wrap" style="max-height:180px;"><table class="result-table"><thead><tr><th>Title</th><th>Code</th></tr></thead><tbody>'
                for it in items[:20]:
                    html_out += f'<tr><td>{esc(it.get("title", ""))}</td><td>{esc(it.get("code", ""))}</td></tr>'
                if len(items) > 20:
                    html_out += f'<tr><td colspan="2"><em>... and {len(items)-20} more</em></td></tr>'
                html_out += '</tbody></table></div>'
    elif tool_name == "get_domain_hierarchy" and isinstance(data, dict) and data.get("lexical"):
        lex = data["lexical"]
        html_out += f'<div style="font-size:0.83rem;">Concept: <strong>{esc(lex.get("title", ""))}</strong></div>'
        for field in ("domainNarrower", "domainBroader", "synonyms"):
            items = lex.get(field, [])
            if items:
                html_out += f'<div style="margin-top:0.3rem;"><strong>{field} ({len(items)}):</strong></div>'
                html_out += '<div class="result-table-wrap" style="max-height:180px;"><table class="result-table"><thead><tr><th>Title</th><th>Code</th></tr></thead><tbody>'
                for it in items[:20]:
                    html_out += f'<tr><td>{esc(it.get("title", ""))}</td><td>{esc(it.get("code", ""))}</td></tr>'
                if len(items) > 20:
                    html_out += f'<tr><td colspan="2"><em>... and {len(items)-20} more</em></td></tr>'
                html_out += '</tbody></table></div>'
    else:
        html_out += f'<pre style="max-height:200px;overflow:auto;font-size:0.72rem;">{esc(json.dumps(data, indent=2)[:1000])}</pre>'

    raw_json = json.dumps(data, indent=2)
    html_out += f'<details class="raw-toggle"><summary>View Response JSON</summary><pre>{esc(raw_json[:3000])}</pre></details>'
    return html_out

def fmt_markdown_to_html(text):
    text = re.sub(r'\[SUMMARY\][\s\S]*?\[/SUMMARY\]', '', text).strip()
    lines = text.split('\n')
    result = []
    in_table = False
    table_rows = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('|') and stripped.endswith('|'):
            if re.match(r'^\|[\s\-:|]+\|$', stripped):
                continue
            if not in_table:
                in_table = True
                table_rows = []
            table_rows.append([c.strip() for c in stripped.split('|')[1:-1]])
        else:
            if in_table:
                result.append(_build_table(table_rows))
                in_table = False
            result.append(stripped)
    if in_table:
        result.append(_build_table(table_rows))
    f = '\n'.join(result)
    f = re.sub(r'^#### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^### (.+)$', r'<h4>\1</h4>', f, flags=re.M)
    f = re.sub(r'^## (.+)$', r'<h3>\1</h3>', f, flags=re.M)
    f = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', f)
    f = re.sub(r'\*(.+?)\*', r'<em>\1</em>', f)
    f = re.sub(r'`(.+?)`', r'<code>\1</code>', f)
    f = f.replace('\n\n', '</p><p>').replace('\n', '<br>')
    return f'<p>{f}</p>'

def _build_table(rows):
    if not rows:
        return ''
    html_t = '<table class="result-table" style="width:100%;margin:0.5rem 0;"><thead><tr>'
    for c in rows[0]:
        html_t += f'<th>{c}</th>'
    html_t += '</tr></thead><tbody>'
    for row in rows[1:]:
        html_t += '<tr>' + ''.join(f'<td>{c}</td>' for c in row) + '</tr>'
    html_t += '</tbody></table>'
    return html_t

def _streaming_dots(msg="Agent is working..."):
    return f'''<div class="streaming-indicator">
        <span class="dot-pulse"></span><span class="dot-pulse"></span><span class="dot-pulse"></span>
        <span style="margin-left:0.3rem;">{esc(msg)}</span>
    </div>'''

def render_agent_run(activity, final_text, label="Final Recommendation", streaming=False):
    normalize_n = sum(1 for a in activity if a["type"] == "tool_call" and is_normalize(a["tool"]))
    kg_n = sum(1 for a in activity if a["type"] == "tool_call" and not is_normalize(a["tool"]))
    count_parts = []
    if normalize_n: count_parts.append(f"{normalize_n} Normalize")
    if kg_n: count_parts.append(f"{kg_n} KG")
    count_label = " · ".join(count_parts) if count_parts else "0 calls"

    norm_idx = 0
    kg_idx = 0
    steps_html = ""
    for a in activity:
        if a["type"] == "reasoning" and a.get("text", "").strip():
            steps_html += f'''
            <div class="step step-reasoning">
                <div class="step-hdr"><span class="step-icon icon-reasoning">\U0001f9e0</span><span class="step-title">Agent Reasoning</span></div>
                <div class="reasoning-text">{fmt_markdown_to_html(a["text"])}</div>
            </div>'''
        elif a["type"] == "tool_call":
            tool = a["tool"]
            norm = is_normalize(tool)
            if norm:
                norm_idx += 1; idx = norm_idx
            else:
                kg_idx += 1; idx = kg_idx
            cls = "step-tool normalize" if norm else "step-tool"
            badge_cls = "badge-normalize" if norm else "badge-kg"
            badge_lbl = "Normalize" if norm else "KG Call"
            icon_cls = "icon-normalize" if norm else "icon-tool"
            icon = "⚙" if norm else "\U0001f50d"
            display_name = TOOL_NAMES.get(tool, tool)
            detail = fmt_tool_input(tool, a.get("args", {}))
            steps_html += f'''
            <div class="step {cls}">
                <span class="step-badge {badge_cls}">{badge_lbl} {idx}</span>
                <div class="step-hdr"><span class="step-icon {icon_cls}">{icon}</span><span class="step-title">{esc(display_name)}</span><span class="step-badge badge-done" style="position:static;margin-left:auto;">Done</span></div>
                <div class="step-detail">{detail}</div>
            </div>'''
        elif a["type"] == "tool_result":
            tool = a["tool"]
            display_name = TOOL_NAMES.get(tool, tool)
            result_html = fmt_tool_result_html(tool, a.get("result", ""))
            steps_html += f'''
            <div class="step step-result">
                <div class="step-hdr"><span class="step-icon icon-result">☰</span><span class="step-title">Results from {esc(display_name)}</span></div>
                <div>{result_html}</div>
            </div>'''

    if final_text:
        bottom = f'''<div class="final-answer">
            <div class="final-label">{esc(label)}</div>
            <div class="final-body">{fmt_markdown_to_html(final_text)}</div>
        </div>'''
    elif streaming:
        bottom = _streaming_dots("Agent is working...")
    else:
        bottom = ""

    if not activity and streaming:
        reasoning_section = _streaming_dots("Agent is thinking...")
    else:
        reasoning_section = f'''<div class="reasoning-steps">
            <div class="reasoning-hdr" onclick="var b=this.parentElement.querySelector('.reasoning-body');b.style.display=b.style.display==='none'?'flex':'none'">
                <h4>Agent Reasoning &amp; API Calls</h4>
                <span class="step-count">{count_label}</span>
                <span class="toggle-icon">▼</span>
            </div>
            <div class="reasoning-body">{steps_html}</div>
        </div>'''

    return f'''{AGENT_CSS}
    <div class="agent-panel">
        <div class="msg-container">
            <div class="msg-avatar">\U0001f916</div>
            <div class="msg-content">
                <div class="msg-header">
                    <span class="msg-sender">Diagnosis Specificity Agent (Med-Based)</span>
                </div>
                {reasoning_section}
                {bottom}
            </div>
        </div>
    </div>'''

print("✓  HTML renderer + streaming engine ready")

✓  HTML renderer + streaming engine ready


## Agent + Streaming Runner

Creates the streaming runner function that parses agent messages into rich HTML activity panels with live updates.

In [5]:
def _extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n".join(parts)
    return str(content) if content else ""

def _build_activity(messages, skip_count=0):
    activity = []
    final_text = ""
    for msg in messages[skip_count:]:
        role = msg.type if hasattr(msg, 'type') else 'unknown'
        if role == 'ai':
            text = _extract_text(msg.content)
            if text.strip():
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    activity.append({"type": "reasoning", "text": text})
                else:
                    final_text = text
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    activity.append({"type": "tool_call", "tool": tc["name"], "args": tc.get("args", {})})
        elif role == 'tool':
            content = msg.content if isinstance(msg.content, str) else json.dumps(msg.content)
            activity.append({"type": "tool_result", "tool": msg.name, "result": content})
    return activity, final_text

def stream_agent(input_messages, display_id, label="Final Recommendation",
                 skip_count=0, recursion_limit=60):
    update_display(HTML(render_agent_run([], "", label=label, streaming=True)), display_id=display_id)

    last_state = None
    for state in agent.stream(
        {"messages": input_messages},
        stream_mode="values",
        config={"recursion_limit": recursion_limit},
    ):
        last_state = state
        activity, final_text = _build_activity(state["messages"], skip_count=skip_count)
        html = render_agent_run(activity, final_text, label=label, streaming=not bool(final_text))
        update_display(HTML(html), display_id=display_id)

    all_messages = last_state["messages"] if last_state else input_messages
    activity, final_text = _build_activity(all_messages, skip_count=skip_count)
    update_display(HTML(render_agent_run(activity, final_text, label=label, streaming=False)), display_id=display_id)
    return activity, final_text, all_messages

print("✓  Agent + streaming runner ready")

✓  Agent + streaming runner ready


## Run: Diabetes + Metformin Example

Diabetes with Metformin — refines to Type 2 diabetes mellitus using medication-guided hierarchy + refinement axes.

In [6]:
CLINICAL_NOTE = """Chief Complaint: Follow-up for diabetes management.

HPI: 55-year-old male with diabetes mellitus diagnosed 3 years ago. Good adherence to medications. Fasting glucose 110-140 mg/dL. No polyuria or polydipsia.

Medications: Metformin 1000 mg BID.
Allergies: NKDA.

Labs: HbA1c 7.2%, Fasting glucose 126 mg/dL.

Assessment and Plan:
1) Diabetes mellitus — HbA1c 7.2% on Metformin. Continue current regimen.

Return in 3 months."""

user_html = f'''{AGENT_CSS}
<div class="agent-panel">
    <div class="msg-container msg-user">
        <div class="msg-avatar">\U0001f464</div>
        <div class="msg-content">
            <div class="msg-header">
                <span class="msg-sender">You</span>
            </div>
            <div class="clinical-note">
                <div class="clinical-note-hdr">
                    <span style="font-size:1.3rem;">\U0001f4cb</span>
                    <span class="clinical-note-title" style="color:#fff;">Clinical Note</span>
                </div>
                <div class="clinical-note-body" style="color:#fff;">{esc(CLINICAL_NOTE)}</div>
            </div>
        </div>
    </div>
</div>'''
display(HTML(user_html))

stream_id = f"refine_{uuid.uuid4().hex[:6]}"
display(HTML(render_agent_run([], "", label="Medication-Based Diagnosis Refinement", streaming=True)), display_id=stream_id)

activity, final_text, all_messages = stream_agent(
    [{"role": "user", "content": CLINICAL_NOTE}],
    display_id=stream_id,
    label="Medication-Based Diagnosis Refinement",
    skip_count=1,
    recursion_limit=60,
)

tc_count = sum(1 for a in activity if a['type'] == 'tool_call')
rs_count = sum(1 for a in activity if a['type'] == 'reasoning')
print(f"✓ Complete. {tc_count} tool calls, {rs_count} reasoning steps.")

✓ Complete. 6 tool calls, 5 reasoning steps.


## Custom Query

Paste your own clinical note below and run the cell.

In [7]:
custom_note = """<paste your clinical note here>"""

# Uncomment below to run:
# user_html2 = f'''{AGENT_CSS}
# <div class="agent-panel"><div class="msg-container msg-user">
#     <div class="msg-avatar">\U0001f464</div>
#     <div class="msg-content"><div class="msg-header"><span class="msg-sender">You</span></div>
#     <div class="clinical-note"><div class="clinical-note-hdr"><span style="font-size:1.3rem;">\U0001f4cb</span>
#     <span class="clinical-note-title" style="color:#fff;">Clinical Note</span></div>
#     <div class="clinical-note-body" style="color:#fff;">{esc(custom_note)}</div></div></div></div></div>'''
# display(HTML(user_html2))
# sid = f"custom_{uuid.uuid4().hex[:6]}"
# display(HTML(render_agent_run([], "", streaming=True)), display_id=sid)
# stream_agent([{"role": "user", "content": custom_note}], display_id=sid, skip_count=1)